Import Libraries

In [1]:
# import libraries
import ee
import geemap
import pandas as pd
import geopandas as gpd

Connect to GEE

In [2]:
# Authenticate GEE
ee.Authenticate()

True

In [3]:
#Initialize GEE
ee.Initialize()

Floods Parameters

In [4]:
start_date = "2022-09-14"
end_date   = "2022-11-18"

before_start = "2022-08-01"
before_end   = "2022-09-10"

flood_threshold = -15
speckle_radius = 50

Define Area of Interest (AOI)

In [5]:
# AOI (GeoJSON)
aoi = ee.Geometry.Polygon(
    [[[6.893233,7.89773],
      [6.666644,7.89773],
      [6.666644,7.683773],
      [6.893233,7.683773],
      [6.893233,7.89773]]]
)

# FAO GAUL Boundaries (Nigeria & Lokoja)

fao_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")

nga_lga = fao_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))

lokoja = nga_lga.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

Download and pre-process Sentinel 1

In [7]:
s1_img_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(start_date, end_date)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filterMetadata("resolution_meters", "equals", 10)
                .filterBounds(aoi)
                .select(["VV", "VH"]))

# image.focal_median(radius, "circle", "meters")

Download Building Dataset

In [8]:
# Load Building Dataset (Google Open Buildings)
buildings = ee.FeatureCollection(
    "GOOGLE/Research/open-buildings-temporal/v1"
)

# Clip buildings to AOI
buildings_aoi = buildings.filterBounds(aoi)

Visualization

In [11]:
import geemap

# Create map
Map = geemap.Map(center=[7.8175, 6.7730], zoom=6)

# Add Nigeria boundaries
Map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
Map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
Map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
Map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")
Map.addLayer(lokoja_by_code, {"color": "red"}, "Lokoja (Code)")

# Add AOI boundary
Map.addLayer(aoi, {"color": "yellow"}, "AOI")

# Display map
Map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…